# Lesson 01: The Forward Corruption Process

In this notebook, we implement the forward process in discrete diffusion — a Markov chain that gradually corrupts clean token sequences. We define transition matrices Q_t and implement both uniform and absorbing-state corruption schedules.

**Paper reference:** D3PM (Austin et al., 2021) Section 3.

In [ ]:
%pip install torch numpy matplotlib --quiet

In [ ]:
import sys
sys.path.insert(0, '../../..')

import torch
import matplotlib.pyplot as plt
from shared.utils.seed import set_seed
from shared.datasets.text import SimpleTokenizer

set_seed(42)

## 1. Understanding Transition Matrices

The forward process is defined by transition matrices $Q_t \in \mathbb{R}^{K \times K}$ where $K$ is the vocabulary size. Each entry $Q_t[i, j]$ gives the probability that token $i$ transitions to token $j$ at timestep $t$.

Let's start with a tiny vocabulary to build intuition.

In [ ]:
from src.forward_process import DiscreteForwardProcess

# Tiny vocabulary: K=5 tokens
# Token 2 is the mask token
vocab_size = 5
T = 100

# Create both types of forward processes
fp_uniform = DiscreteForwardProcess(vocab_size, T, schedule="uniform", mask_token_id=2)
fp_absorbing = DiscreteForwardProcess(vocab_size, T, schedule="absorbing", mask_token_id=2)

In [ ]:
# Look at the single-step transition matrix at t=1
Q1_uniform = fp_uniform.get_qt_matrix(1)
print("Uniform Q_1 (single step):")
print(Q1_uniform.numpy().round(4))
print()

Q1_absorbing = fp_absorbing.get_qt_matrix(1)
print("Absorbing Q_1 (single step):")
print(Q1_absorbing.numpy().round(4))

# Verify rows sum to 1 (valid probability distributions)
assert torch.allclose(Q1_uniform.sum(dim=1), torch.ones(vocab_size), atol=1e-6)
assert torch.allclose(Q1_absorbing.sum(dim=1), torch.ones(vocab_size), atol=1e-6)
print("\nAll rows sum to 1.")

## 2. Cumulative Corruption: $\bar{Q}_t$

The cumulative matrix $\bar{Q}_t = Q_1 Q_2 \cdots Q_t$ describes the total corruption from time 0 to time $t$. We use a closed-form expression instead of multiplying matrices.

Let $\bar{\alpha}_t = \prod_{s=1}^{t} (1 - \beta_s)$ be the probability a token is *unchanged*.

In [ ]:
# Look at cumulative corruption at different timesteps
for t in [1, T // 4, T // 2, T]:
    Qt_bar = fp_absorbing.get_qt_bar(t)
    corruption_rate = fp_absorbing.get_corruption_rate(t)
    print(f"t={t:3d}: corruption rate = {corruption_rate:.4f}")
    print(f"  Row 0 of Q_bar_t: {Qt_bar[0].numpy().round(4)}")
    print()

In [ ]:
# At t=T, absorbing schedule should send everything to the mask token
Qt_bar_T = fp_absorbing.get_qt_bar(T)
# Each row should have most mass on the mask token (column 2)
print(f"Q_bar_T row 0: {Qt_bar_T[0].numpy().round(4)}")
print(f"Mass on mask token: {Qt_bar_T[0, 2].item():.4f}")
print(f"Mass remaining on original token: {Qt_bar_T[0, 0].item():.6f}")

## 3. Sampling Corrupted Sequences

The `sample_q_t` method lets us draw $x_t \sim q(x_t | x_0)$ directly, without simulating intermediate steps. This is essential for training: we need to quickly generate corrupted versions of our training data.

In [ ]:
# Create a small tokenizer and encode a sentence
texts = [
    "the cat sat on the mat",
    "a dog ran in the park",
    "the sun is bright today",
]
tokenizer = SimpleTokenizer(texts, level="char")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Mask token ID: {tokenizer.mask_id}")

# Encode a sentence
sentence = "the cat sat on the mat"
x_0 = torch.tensor(tokenizer.encode(sentence)).unsqueeze(0)  # (1, seq_len)
print(f"\nOriginal: '{sentence}'")
print(f"Token IDs: {x_0[0].tolist()}")

In [ ]:
# Create forward processes matching our tokenizer's vocabulary
T = 1000
fp = DiscreteForwardProcess(
    vocab_size=tokenizer.vocab_size,
    num_timesteps=T,
    schedule="absorbing",
    mask_token_id=tokenizer.mask_id,
)

# Visualize corruption at t=0, T/4, T/2, T
print("Absorbing corruption over time:")
print(f"t=   0: {sentence}")
for t_val in [T // 4, T // 2, 3 * T // 4, T]:
    t = torch.tensor([t_val])
    x_t = fp.sample_q_t(x_0, t)
    decoded = tokenizer.decode(x_t[0].tolist())
    rate = fp.get_corruption_rate(t_val)
    print(f"t={t_val:4d} ({rate:.0%} corrupt): {decoded}")

In [ ]:
# Same thing with uniform corruption
fp_uni = DiscreteForwardProcess(
    vocab_size=tokenizer.vocab_size,
    num_timesteps=T,
    schedule="uniform",
)

print("Uniform corruption over time:")
print(f"t=   0: {sentence}")
for t_val in [T // 4, T // 2, 3 * T // 4, T]:
    t = torch.tensor([t_val])
    x_t = fp_uni.sample_q_t(x_0, t)
    decoded = tokenizer.decode(x_t[0].tolist())
    rate = fp_uni.get_corruption_rate(t_val)
    print(f"t={t_val:4d} ({rate:.0%} corrupt): {decoded}")

## 4. Plotting the Corruption Rate

Let's visualize how the corruption rate $1 - \bar{\alpha}_t$ changes over time.

In [ ]:
timesteps = list(range(1, T + 1))
corruption_rates = [fp.get_corruption_rate(t) for t in timesteps]

plt.figure(figsize=(8, 4))
plt.plot(timesteps, corruption_rates)
plt.xlabel('Timestep t')
plt.ylabel('Corruption Rate (1 - alpha_bar_t)')
plt.title('Forward Process: Corruption Rate Over Time')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Verifying Correctness

Let's verify that our closed-form $\bar{Q}_t$ matches the matrix product $Q_1 Q_2 \cdots Q_t$.

In [ ]:
# Use small vocab for matrix multiplication check
K = 5
T_small = 20
fp_check = DiscreteForwardProcess(K, T_small, schedule="absorbing", mask_token_id=2)

# Compute Q_bar via sequential multiplication
Q_bar_product = torch.eye(K)
for t in range(1, T_small + 1):
    Q_bar_product = Q_bar_product @ fp_check.get_qt_matrix(t)

# Compare with closed-form
Q_bar_closed = fp_check.get_qt_bar(T_small)

print("Q_bar via matrix products:")
print(Q_bar_product.numpy().round(4))
print("\nQ_bar via closed form:")
print(Q_bar_closed.numpy().round(4))
print(f"\nMax difference: {(Q_bar_product - Q_bar_closed).abs().max().item():.6f}")

assert torch.allclose(Q_bar_product, Q_bar_closed, atol=1e-4), "Mismatch!"
print("Closed-form matches matrix product.")

## Exercises

1. **Verify the absorbing property:** At `t=T`, verify that `Q_bar_T` sends all mass to the mask token.

2. **Compare schedules:** Corrupt the same sentence with both schedules at `t=T/2`. Which is more readable?

3. **Cosine schedule:** Modify `_compute_beta_schedule` to use a cosine schedule and compare the corruption rate curves.

## What's Next

In Lesson 02, we learn to **undo** this corruption by computing the reverse (denoising) step.